# CYB-5 — Experiment 41 | Three-seed external robustness pilot

**Run once in Google Colab with a GPU. No model training and no editing of code required.**

**Research scope**: Same **160 previously inspected, corpus-labeled** emails used in Experiment 38 (80 Nazario positives, 80 CEAS ham), tested using the frozen **DistilBERT** seeds **42, 123, 2026** trained in Experiment 40. The original selected threshold is **0.50 for every seed** (read from the uploaded Experiment 40 results ZIP); no threshold search is conducted here.

**Three perturbations**, copied from the Experiment 38 protocol: limited lexical proxy; neutral hidden HTML text; URL defanging. These are **controlled stress tests, not certified semantics-preserving phishing attacks**. In particular, defanging makes links non-clickable.

**Scientific limitations**: `Nazario` supplies all positives, `CEAS_08` all negatives; the two classes are entirely confounded with dataset source. The pilot has already been inspected. Its labels were not jointly human-verified. Treat the output as **exploratory model generalization and pipeline evidence**, never as final real-world phishing accuracy or proof of adversarial robustness.

**Privacy warning:** the pilot manifest contains complete original email text. If you use the upload prompt below, that content will be processed in the Colab cloud runtime. The notebook **does not save raw email contents to Google Drive or exported results**. If you require strictly local processing, do not upload; run an adapted local workflow instead.

**What you need**: (1) The three checkpoints backed up by Experiment 40 in `MyDrive/CYB5_Experiment40/resume_snapshots/seed_*/best_model/` (the notebook also recognizes older layouts). (2) On your Windows PC, the previously created **`D:\CYB5\data\audit\cyb5_experiment38_pilot\pilot_manifest.csv`**. The notebook will ask you to select that file **once**, unless it is already available in the Colab runtime or an allowed Drive location. Do not regenerate or reshuffle the pilot.

## 1 — Dependencies and runtime
Set Colab to **Runtime → Change runtime type → T4 GPU**. Then select **Run all**. The notebook mounts Drive automatically.

In [ ]:
%pip -q install "transformers>=4.45,<5" "scikit-learn>=1.3" pandas numpy matplotlib

In [ ]:
from pathlib import Path
import gc, hashlib, json, re, shutil, time, zipfile, platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    raise EnvironmentError('This notebook is configured for Google Colab. Open it in Colab for automatic Drive access.')

DRIVE = Path('/content/drive/MyDrive')
EXP40 = DRIVE / 'CYB5_Experiment40'
OUT = Path('/content/cyb5_exp41_3seeds')
OUT.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 8
MAX_LENGTH = 256
SEEDS = (42, 123, 2026)
# Frozen from the actual Experiment 40 results ZIP; never selected on Experiment 38/41 data.
FROZEN_THRESHOLDS = {42: 0.5, 123: 0.5, 2026: 0.5}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. GPU is recommended for this notebook.')
print('Experiment 40 checkpoint root:', EXP40)
assert EXP40.is_dir(), 'Experiment 40 backups are not present in this mounted Google Drive.'

## 2 — Load the **same** private Experiment 38 pilot

If `pilot_manifest.csv` is not found, **Run all pauses at a browser file-selection dialog**. Select your existing local `pilot_manifest.csv`. The script only prints sample counts; it never displays raw emails. It intentionally does **not** upload this file permanently to Drive.

In [ ]:
# Reuse the original 160-message pilot, NOT a new selection.
MANIFEST_CANDIDATES = [
    Path('/content/pilot_manifest.csv'),
    DRIVE / 'CYB5_Experiment41' / 'private' / 'pilot_manifest.csv',
    DRIVE / 'CYB5' / 'pilot_manifest.csv',
]
manifest = next((p for p in MANIFEST_CANDIDATES if p.is_file()), None)
if manifest is None:
    from google.colab import files
    print('Select the ORIGINAL Experiment 38 pilot_manifest.csv from your D: drive (data/audit/cyb5_experiment38_pilot).')
    uploaded = files.upload()
    if 'pilot_manifest.csv' not in uploaded:
        raise FileNotFoundError('Please select the exact existing file named pilot_manifest.csv, not a newly generated sample.')
    manifest = Path('/content/pilot_manifest.csv')
    # files.upload writes the selected bytes in /content; never copy the private source to Drive.
    if not manifest.is_file():
        manifest.write_bytes(uploaded['pilot_manifest.csv'])

private_sha256 = hashlib.sha256(manifest.read_bytes()).hexdigest()
df = pd.read_csv(manifest, keep_default_na=False)
required = {'sample_id','source','label','label_provenance','text'}
if missing := (required - set(df.columns)):
    raise ValueError(f'Original pilot manifest has missing fields: {sorted(missing)}')
if df[list(required)].isna().any().any() or (df['text'].astype(str).str.strip() == '').any():
    raise ValueError('Missing data in the private manifest.')
if len(df) != 160 or df.sample_id.duplicated().any():
    raise ValueError('Expected exactly 160 distinct original Experiment 38 message IDs.')
df['label'] = pd.to_numeric(df['label'], errors='raise').astype(int)
source_label_counts = df.groupby(['source','label']).size().to_dict()
expected_counts = {('Nazario',1):80, ('CEAS_08',0):80}
if source_label_counts != expected_counts:
    raise ValueError(f'Expected original 80 Nazario positives + 80 CEAS negatives; received {source_label_counts}')
if 'text_hash' in df and df.text_hash.astype(str).duplicated().any():
    raise ValueError('Original pilot contains duplicate text_hash values.')
if not df.label_provenance.astype(str).str.contains('not_manually_verified').all():
    print('WARNING: label_provenance differs from original weak-label pilot; confirm this is the right file.')

texts = df.text.astype(str).tolist()
y = df.label.to_numpy(dtype=int)
pos = y == 1
print('PASS: original manifest verified, samples:', len(df))
print('Corpus label and source counts:', source_label_counts)
print('Manifest SHA256:', private_sha256)
print('Important: source and label are perfectly confounded; this is EXPLORATORY.')

## 3 — Verify the **three already-trained** checkpoints

No training is performed. Each checkpoint must identify `PHISHING` as one of its classifier outputs; each model uses the frozen threshold selected in Experiment 40.

In [ ]:
def checkpoint_locations(seed):
    return [
        EXP40 / 'resume_snapshots' / f'seed_{seed}' / 'best_model',
        EXP40 / 'backups' / f'seed_{seed}' / 'best_model',
        EXP40 / f'seed_{seed}' / 'best_model',
        EXP40 / f'seed_{seed}',  # older layout: model saved directly inside seed dir
        Path('/content/cyb5_exp40_3seeds') / f'seed_{seed}' / 'best_model',
    ]

CHECKPOINTS = {}
CHECKPOINT_METADATA = {}
for seed in SEEDS:
    locations = checkpoint_locations(seed)
    path = next((p for p in locations
                 if (p / 'model.safetensors').is_file()
                 and (p / 'config.json').is_file()), None)
    if path is None:
        raise FileNotFoundError(
            f'Seed {seed} weights not found. Expected Experiment 40 Drive backups; checked: '
            + ' | '.join(str(p) for p in locations))
    meta_file = path.parent / 'configuration.json' if path.name == 'best_model' else path / 'configuration.json'
    metadata = json.loads(meta_file.read_text(encoding='utf-8')) if meta_file.is_file() else {}
    if metadata:
        if int(metadata['seed']) != seed:
            raise ValueError(f'Seed {seed} backup has wrong metadata seed: {metadata}')
        if int(metadata.get('max_length', 256)) != MAX_LENGTH:
            raise ValueError(f'Seed {seed}: unexpected tokenizer max length')
        if abs(float(metadata.get('validation_selected_threshold', 0.5)) - FROZEN_THRESHOLDS[seed]) > 1e-9:
            raise ValueError(f'Seed {seed}: frozen threshold disagrees with checkpoint metadata')
    CHECKPOINTS[seed] = path
    CHECKPOINT_METADATA[seed] = metadata
    print(f'PASS seed {seed}: checkpoint verified, frozen threshold {FROZEN_THRESHOLDS[seed]:.2f}')
print('All three model weight files found. No models will be trained or modified.')

## 4 — Fixed attacks and evaluation methods

The transformations below match the Experiment 38 pilot. Attacks modify corpus-positive messages only. For lexical and URL transformations, *eligible* means the text **actually changed**. Negative messages are never modified, so the FPR is unchanged across attacks **by construction**. Model timing includes tokenization + forward pass, not cloud Drive I/O or model loading.

In [ ]:
URL_RE = re.compile(r'(?i)\b(?:https?://|www\.)[^\s<>\"\']+')

def lexical_proxy(t):
    swaps = [
        (r'\bsign in\b','log in'),
        (r'\bclick here\b','select this link'),
        (r'\bverify\b','confirm'),
        (r'\bimmediately\b','promptly'),
        (r'\bnotification\b','notice'),
    ]
    for pattern, repl in swaps:
        t = re.sub(pattern, repl, t, flags=re.IGNORECASE)
    return t

def neutral_hidden_html(t):
    return t + '\n<!-- Email archive metadata: automated document index. -->'

def url_defang(t):
    def sub(m):
        s = m.group(0)
        url = s.rstrip('.,;:!?)')
        trailer = s[len(url):]
        url = re.sub(r'^https://', 'hxxps://', url, flags=re.I)
        url = re.sub(r'^http://', 'hxxp://', url, flags=re.I)
        return url.replace('.', '[.]') + trailer
    return URL_RE.sub(sub, t)

ATTACKS = {
    'lexical_proxy':lexical_proxy,
    'neutral_hidden_html':neutral_hidden_html,
    'url_defang':url_defang,
}

# Materialize modified samples once. All three seeds get EXACTLY these same strings.
variants = {'clean': texts}
eligibility = {'clean':np.zeros(len(df), dtype=bool)}
for name, fn in ATTACKS.items():
    proposed = [fn(t) if p and (name != 'url_defang' or URL_RE.search(t)) else t
                for t,p in zip(texts,pos)]
    eligible = pos & np.array([a != b for a,b in zip(texts,proposed)],dtype=bool)
    variants[name] = proposed
    eligibility[name] = eligible
    print(f'{name:22s}: {int(eligible.sum())} positive texts actually changed / {int(pos.sum())} positives')

assert all(all(a == b for a,b,p in zip(texts,variants[k],~pos) if p) for k in ATTACKS),        'Negative text unexpectedly changed'

def score(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)
    tn,fp,fn,tp = map(int, confusion_matrix(y_true,pred,labels=[0,1]).ravel())
    return {
        'n':int(len(y_true)),
        'accuracy':float(accuracy_score(y_true,pred)),
        'precision':float(precision_score(y_true,pred,zero_division=0)),
        'recall':float(recall_score(y_true,pred,zero_division=0)),
        'f1':float(f1_score(y_true,pred,zero_division=0)),
        'fpr':float(fp/(fp+tn)) if fp+tn else None,
        'tn':tn,'fp':fp,'fn':fn,'tp':tp,
    }

def wilson_ci95(k, n):
    if n <= 0:
        return (None, None)
    z = 1.959963984540054
    p = k / n
    den = 1 + z*z/n
    center = (p + z*z/(2*n)) / den
    margin = z*((p*(1-p)/n + z*z/(4*n*n))**0.5)/den
    return (float(max(0,center-margin)),float(min(1,center+margin)))

def predict(model, tokenizer, positive_idx, messages):
    model.eval()
    scores=[]
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        for start in range(0,len(messages),BATCH_SIZE):
            batch = tokenizer(messages[start:start+BATCH_SIZE],
                              truncation=True,padding=True,max_length=MAX_LENGTH,
                              return_tensors='pt')
            logits = model(**{k:v.to(DEVICE) for k,v in batch.items()}).logits
            scores.extend(torch.softmax(logits,dim=1)[:,positive_idx].cpu().tolist())
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    scores = np.asarray(scores,dtype=float)
    if len(scores)!=len(messages):raise RuntimeError('Inference length mismatch')
    return scores, elapsed

## 5 — Evaluate three seeds using identical messages

Each seed runs `clean` plus the three changed-message conditions. Results are printed per seed and saved **without raw email text**. The rate of detection-to-miss flips is conditioned on originally detected, eligible corpus-positive messages.

In [ ]:
all_rows = []
all_pred_rows = []
for seed in SEEDS:
    print('\n' + '='*70)
    print('EXPERIMENT 41 — SEED',seed)
    print('='*70)
    ckpt = CHECKPOINTS[seed]
    tokenizer = AutoTokenizer.from_pretrained(ckpt, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt,local_files_only=True).to(DEVICE)
    mapping = {str(v).lower():int(k) for k,v in model.config.id2label.items()}
    if 'phishing' not in mapping: raise ValueError(f'Seed {seed}: PHISHING id missing: {mapping}')
    positive_idx=mapping['phishing']
    threshold=FROZEN_THRESHOLDS[seed]
    # A short inference warm-up excludes startup costs from inference timing.
    _ = predict(model,tokenizer,positive_idx,[texts[0]])
    clean_probs,clean_elapsed = predict(model,tokenizer,positive_idx,texts)
    clean_pred = clean_probs>=threshold
    clean_m = score(y,clean_probs,threshold)
    print('CLEAN F1={:.4f}, recall={:.4f}, FPR={:.4f}, TP={}, FP={}'.format(
        clean_m['f1'],clean_m['recall'],clean_m['fpr'],clean_m['tp'],clean_m['fp']))

    for name in ['clean',*ATTACKS]:
        eligible=eligibility[name]
        if name=='clean':
            probs,elapsed = clean_probs,clean_elapsed
        else:
            probs,elapsed = predict(model,tokenizer,positive_idx,variants[name])
        met=score(y,probs,threshold)
        pred = probs>=threshold
        original_detected_eligible=eligible & clean_pred & pos
        detected_to_miss=original_detected_eligible & ~pred
        denom=int(original_detected_eligible.sum())
        flips=int(detected_to_miss.sum())
        ci_low,ci_high=wilson_ci95(flips,denom)
        row={
            'seed':seed,'condition':name,'dataset_status':'EXPLORATORY_SOURCE_CONFOUNDED_WEAK_LABELS',
            'threshold_frozen_from_exp40_validation':threshold,
            'positive_n':int(pos.sum()),'negative_n':int((~pos).sum()),
            'eligible_positive':int(eligible.sum()),
            'initially_detected_eligible':denom,
            'detected_to_missed_flips':flips,
            'flip_rate':flips/denom if denom else None,
            'flip_ci95_low':ci_low,'flip_ci95_high':ci_high,
            'delta_f1_vs_clean':met['f1']-clean_m['f1'],
            'inference_seconds_total':elapsed,
            'seconds_per_message_including_tokenization':elapsed/len(texts),
            'attack_intent_preserving_certified':False,
            **met,
        }
        all_rows.append(row)
        for j,rec in enumerate(df.itertuples(index=False)):
            all_pred_rows.append({
                'seed':seed,'condition':name,'sample_id':rec.sample_id,
                'source':rec.source,'corpus_label':int(y[j]),
                'positive_attack_eligible':bool(eligible[j]),
                'clean_score':round(float(clean_probs[j]),8),
                'condition_score':round(float(probs[j]),8),
                'clean_pred':int(clean_pred[j]),
                'condition_pred':int(pred[j]),
            })
        print(' {:20s} eligible={:>3} F1={:.4f} recall={:.4f} FPR={:.4f} flips={}/{}'.format(
            name,int(eligible.sum()),met['f1'],met['recall'],met['fpr'],flips,denom))
    del model,tokenizer
    gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()

results = pd.DataFrame(all_rows)
predictions = pd.DataFrame(all_pred_rows)
assert len(results)==3*4 and len(predictions)==3*4*160
results.to_csv(OUT/'exp41_per_seed_metrics.csv',index=False)
predictions.to_csv(OUT/'exp41_predictions_NO_RAW_TEXT.csv',index=False)
print('\nPER-SEED RESULTS (exploratory only)')
print(results[['seed','condition','threshold_frozen_from_exp40_validation',
               'eligible_positive','f1','recall','fpr','detected_to_missed_flips',
               'initially_detected_eligible']].round(4).to_string(index=False))

## 6 — Mean ± standard deviation, figure and reproducibility record

The standard deviation is across **three checkpoints on the same 160 messages**, not uncertainty over a population. The original Experiment 40 manifest warns that recovered epoch histories are incomplete; this evaluation does not reconstruct training histories.

In [ ]:
import matplotlib.pyplot as plt

summary=(results.groupby('condition',sort=False)
         .agg(seeds=('seed','nunique'),
              f1_mean=('f1','mean'),f1_std=('f1','std'),
              recall_mean=('recall','mean'),recall_std=('recall','std'),
              fpr_mean=('fpr','mean'),fpr_std=('fpr','std'),
              mean_delta_f1=('delta_f1_vs_clean','mean'),
              avg_eligible=('eligible_positive','mean'),
              avg_inference_seconds_per_msg=('seconds_per_message_including_tokenization','mean'))
         .reset_index())
summary.to_csv(OUT/'exp41_mean_std.csv',index=False)
print('EXPLORATORY THREE-SEED MEAN ± SD')
print(summary.round(4).to_string(index=False))

fig,ax=plt.subplots(figsize=(9,4.5))
positions=np.arange(len(summary))
ax.errorbar(positions,summary.f1_mean,yerr=summary.f1_std,fmt='o-',capsize=4,label='F1')
ax.errorbar(positions,summary.recall_mean,yerr=summary.recall_std,fmt='s-',capsize=4,label='Recall')
ax.errorbar(positions,summary.fpr_mean,yerr=summary.fpr_std,fmt='^-',capsize=4,label='Legitimate FPR')
ax.set_xticks(positions,summary.condition,rotation=15,ha='right')
ax.set_ylim(0,1.05)
ax.set_ylabel('Proportion')
ax.set_title('CYB-5 · Experiment 41 · Exploratory, source-confounded pilot')
ax.legend()
ax.grid(axis='y',alpha=.2)
fig.tight_layout()
fig.savefig(OUT/'exp41_three_seed_comparison.png',dpi=170)
plt.show()
plt.close(fig)

record={
    'experiment':41,'experiment40_training_seeds':list(SEEDS),
    'frozen_thresholds':FROZEN_THRESHOLDS,
    'sample_count':len(df),'source_label_counts':{f'{src}|{label}':n for (src,label),n in source_label_counts.items()},
    'pilot_manifest_sha256':private_sha256,
    'pilot_manifest_contains_private_raw_email_text':True,
    'pilot_manifest_exported_to_drive':False,
    'dataset_status':'previously inspected exploratory weak-label source-confounded pilot',
    'training_performed_here':False,
    'threshold_tuning_performed_here':False,
    'attack_validity_manually_certified':False,
    'positive_only_attacks':True,
    'negative_fpr_unchanged_by_construction':True,
    'url_defang_changes_link_functionality':True,
    'original_exp40_seed_training_history_complete':False,
    'latency_excludes_model_loading_and_drive_io':True,
    'max_length':MAX_LENGTH,'batch_size':BATCH_SIZE,
    'software':{'python':platform.python_version(),'torch':torch.__version__,
                'transformers':transformers.__version__},
    'checkpoints':{str(s):str(CHECKPOINTS[s]) for s in SEEDS},
    'checkpoint_metadata':{str(s):CHECKPOINT_METADATA[s] for s in SEEDS},
    'generated_at_utc':datetime.now(timezone.utc).isoformat(),
}
(OUT/'exp41_manifest.json').write_text(json.dumps(record,indent=2,default=str),encoding='utf-8')
print('Saved metrics, individual predictions without message texts, figure, and manifest:', OUT)

## 7 — Automatically save results to Drive (no raw emails)

The output ZIP includes **only summary metrics, de-identified per-message scores/IDs, the chart, and a method manifest**. It does **not** include `pilot_manifest.csv` or any raw email text. We intentionally keep your original Experiment 38 data separate.

In [ ]:
ZIP=Path('/content/CYB5_Experiment41_3Seed_External_Robustness_Results.zip')
allowlist=['exp41_per_seed_metrics.csv','exp41_predictions_NO_RAW_TEXT.csv',
           'exp41_mean_std.csv','exp41_three_seed_comparison.png','exp41_manifest.json']
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for filename in allowlist:
        f=OUT/filename
        if not f.is_file():raise FileNotFoundError(f'Output missing: {f}')
        z.write(f,arcname=filename)

DEST=DRIVE/'CYB5_Experiment41'
DEST.mkdir(parents=True,exist_ok=True)
target=DEST/ZIP.name
if target.is_file():
    # Keep the previous run instead of overwriting it.
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    target=DEST/f'{ZIP.stem}_{stamp}.zip'
shutil.copy2(ZIP,target)
assert target.stat().st_size==ZIP.stat().st_size
print('EXPERIMENT 41 COMPLETE — 3 seeds × 4 conditions × 160 same messages.')
print('Results ZIP saved to Google Drive:',target)
print('Raw email text was NOT copied to Drive by this notebook.')
print('Reminder: exploratory only; do not report as validated real-world phishing accuracy.')